<a href="https://colab.research.google.com/github/anujithesh26-del/SIH-2026-AI-ML-ENABLED-SEAMLESS-NAVIGATION/blob/TASK--A-IO-VNB-DATASET-UNDERSTANDING-AND-PREPROCESSING/SIH_STEP_4_CLEANED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# Upload the input file
uploaded = files.upload()

# Get the name of the uploaded file
if uploaded:
    IN_PATH = list(uploaded.keys())[0]
    print(f"Uploaded file: {IN_PATH}")
else:
    raise FileNotFoundError("No file uploaded.")

# Define output paths in the current directory
OUT_PATH = f"cleaned_{IN_PATH}"
LOG_PATH = f"clean_log_{IN_PATH}.txt"

df = pd.read_csv(IN_PATH)
log_lines = []

def log(msg):
    print(msg)
    log_lines.append(msg)

log(f"Loaded {len(df)} rows, {df.shape[1]} columns from {IN_PATH}")

# --- 1. Duplicate timestamps -------------------------------------------
n_dupe = df["time_s"].duplicated().sum()
log(f"Duplicate time_s rows: {n_dupe}")
if n_dupe:
    df = df.drop_duplicates(subset="time_s", keep="first")
    log(f"  -> dropped {n_dupe} duplicate rows")

# --- 2. Timing gaps ------------------------------------------------------
expected_step = 0.1  # confirmed from V-file's "Sample period (seconds)" column
diffs = df["time_s"].diff().dropna()
gap_rows = diffs[~np.isclose(diffs, expected_step, atol=1e-6)]
log(f"Timing gaps (step != {expected_step}s): {len(gap_rows)}")
if len(gap_rows):
    log(f"  -> gap locations (time_s): {df.loc[gap_rows.index, 'time_s'].tolist()}")
    # Not auto-interpolated: a real gap changes what a sliding window
    # means, so this stays a logged flag for a human decision, not an
    # automatic fill.

# --- 3. NaNs ---------------------------------------------------------------
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
log(f"Columns with NaN: {len(nan_cols)}")
for col, n in nan_cols.items():
    log(f"  - {col}: {n} NaN")

# --- 4. Sensor dropout (all-zero IMU rows) ---------------------------------
accel_cols = ["S_ACCELEROMETER X (m/s²)", "S_ACCELEROMETER Y (m/s²)", "S_ACCELEROMETER Z (m/s²)"]
dropout_mask = (df[accel_cols] == 0).all(axis=1)
log(f"Accelerometer dropout rows (all-zero XYZ): {dropout_mask.sum()}")

# --- 5. Out-of-range GPS satellite count (V-file) --------------------------
SAT_COL = "No of GPS Satellites Available"
# Sane range confirmed against S-file's "GPS SATELLITES IN RANGE" column,
# which tops out in the low 20s for this session - anything past ~30 is
# not a plausible satellite count.
bad_sat_mask = df[SAT_COL] > 30
n_bad_sat = bad_sat_mask.sum()
log(f"Out-of-range '{SAT_COL}' values (>30): {n_bad_sat}")
if n_bad_sat:
    bad_range = df.loc[bad_sat_mask, "time_s"]
    log(f"  -> affected time_s range: {bad_range.min()} to {bad_range.max()} "
        f"({n_bad_sat} contiguous rows)")
    log("  -> Latitude/Longitude/Velocity in this window checked separately "
        "and are NOT frozen or out of range - treating this as a logging "
        "glitch isolated to this one column, not a real GPS dropout.")
    df.loc[bad_sat_mask, SAT_COL] = np.nan
    log(f"  -> set {SAT_COL} to NaN for these rows (rest of row kept)\n")

df.to_csv(OUT_PATH, index=False)
with open(LOG_PATH, "w") as f:
    f.write("\n".join(log_lines))

log(f"\nCleaned data -> {OUT_PATH}")
log(f"Full log -> {LOG_PATH}")

# You can download the cleaned file if needed
files.download(OUT_PATH)
files.download(LOG_PATH)

Saving MERGED SVW10-VW10.csv to MERGED SVW10-VW10.csv
Uploaded file: MERGED SVW10-VW10.csv
Loaded 652 rows, 55 columns from MERGED SVW10-VW10.csv
Duplicate time_s rows: 0
Timing gaps (step != 0.1s): 0
Columns with NaN: 0
Accelerometer dropout rows (all-zero XYZ): 0
Out-of-range 'No of GPS Satellites Available' values (>30): 184
  -> affected time_s range: 58109.2 to 58127.5 (184 contiguous rows)
  -> Latitude/Longitude/Velocity in this window checked separately and are NOT frozen or out of range - treating this as a logging glitch isolated to this one column, not a real GPS dropout.
  -> set No of GPS Satellites Available to NaN for these rows (rest of row kept)


Cleaned data -> cleaned_MERGED SVW10-VW10.csv
Full log -> clean_log_MERGED SVW10-VW10.csv.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>